### Agent ReAct Chat - Practice & Exercises

In this notebook, you will practice building structured chat agents with conversation memory.

#### Part 1: Follow-Along Practice

##### Goal / Question
Create a conversational ReAct agent with access to Wikipedia and Time tools, preserving chat memory context across interactions.

##### Sample Input
```python
"User: Who was Albert Einstein?"
"User: What time is it?"
```

##### Sample Output
Responses providing facts and time using tools while keeping memory context.

##### Plan / Steps
1. Define Time and Wikipedia query functions.
2. Wrap functions in `Tool` classes.
3. Pull prompt `hwchase17/structured-chat-agent` from hub.
4. Initialize model using `init_chat_model("llama-3.3-70b-versatile", model_provider="groq")`.
5. Initialize `ConversationBufferMemory(memory_key="chat_history", return_messages=True)`.
6. Create structured chat agent using `create_structured_chat_agent` and execute it.

In [ ]:
from dotenv import load_dotenv
from langchain import hub
from langchain.agents import AgentExecutor, create_structured_chat_agent
from langchain.memory import ConversationBufferMemory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import Tool
from langchain.chat_models import init_chat_model
import datetime
from wikipedia import summary

load_dotenv()
model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

def get_time(*args, **kwargs):
    return datetime.datetime.now().strftime("%I:%M %p")

def search_wiki(query):
    try:
        return summary(query, sentences=2)
    except:
        return "Not found"

tools = [
    Tool(name="Time", func=get_time, description="Get current time"),
    Tool(name="Wikipedia", func=search_wiki, description="Search info")
]
prompt = hub.pull("hwchase17/structured-chat-agent")
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
agent = create_structured_chat_agent(llm=model, tools=tools, prompt=prompt)
agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True, memory=memory, handle_parsing_errors=True)

print(agent_executor.invoke({"input": "Who was Albert Einstein?"}))

#### Part 2: Practice Exercises

##### Exercise: Add Math Tool to Chat Agent

##### Goal / Question
Add a custom multiplication tool `Multiply` to the structured chat agent and verify that the agent can answer math questions and general conversation in a multi-turn session.

##### Sample Input
```python
"User: Multiply 5 and 6"
```

##### Sample Output
The product of 5 and 6 is 30.

##### Plan / Steps
1. Define a multiplication function `multiply_vals(query: str)` that extracts two numbers and multiplies them.
2. Map it to a `Tool` constructor.
3. Register it in the tools list alongside `Time` and `Wikipedia`.
4. Execute the agent executor to verify multiplication query.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
def multiply_vals(query: str, *args, **kwargs):
    nums = [float(s) for s in query.split() if s.replace(".", "", 1).isdigit()]
    return str(nums[0] * nums[1]) if len(nums) >= 2 else "Invalid numbers"

math_tool = Tool(name="Multiply", func=multiply_vals, description="Multiply two space-separated numbers.")
extended_tools = tools + [math_tool]
extended_agent = create_structured_chat_agent(llm=model, tools=extended_tools, prompt=prompt)
extended_executor = AgentExecutor.from_agent_and_tools(
    agent=extended_agent, tools=extended_tools, verbose=True, memory=memory, handle_parsing_errors=True
)
print(extended_executor.invoke({"input": "Multiply 5 and 6"}))
```
</details>